# Experimento H2 - Catastrophe Analysis - Grupo B
## Brazo `A0_cero`

> **Hipotesis H2.** La mejora en la ganancia proviene de senalarle al modelo que
> el dato falta, no de reconstruir su valor.

| | |
|---|---|
| **brazo** | `A0_cero` |
| **tratamiento** | Ningun tratamiento (deja los ceros) |
| **senala la ausencia** | NO |
| **reconstruye el valor** | NO |
| **experimento** | `WF6300` |
| **semillas** | 100043, 200063, 300089, 500069, 700021, 181219, 410341 |
| **future** | 202107 (Grupo B) |
| **metrica** | `ganancia_suavizada_max` de cada `PARAM_<semilla>.yml` |

**Lo unico que cambia entre los 5 brazos es la celda de Catastrophe Analysis.**
Todo lo demas -- FE historico, training strategy, hiperparametros, final train,
scoring -- es identico.

### 6.1 Objetivo

Presentar un workflow/pipeline completo al que los estudiantes deberán enriquecer

#### 6.2  Seteo del ambiente en Google Colab

Esta parte se debe correr con el runtime en Python3
<br>Ir al menu, Runtime -> Change Runtime Type -> Runtime type ->  **Python 3**

Conectar la virtual machine donde esta corriendo Google Colab con el  Google Drive, para poder tener persistencia de archivos

In [ ]:
# primero establecer el Runtime de Python 3
from google.colab import drive
drive.mount('/content/.drive')

Para correr la siguiente celda es fundamental en Arranque en Frio haber copiado el archivo kaggle.json al Google Drive, en la carpeta indicada en el instructivo

<br>los siguientes comando estan en shell script de Linux
*   Crear las carpetas en el Google Drive
*   "instalar" el archivo kaggle.json desde el Google Drive a la virtual machine para que pueda ser utilizado por la libreria  kaggle de Python
*   Bajar el  **dataset_pequeno**  al  Google Drive  y tambien al disco local de la virtual machine que esta corriendo Google Colab
*   Bajar el **dataset_historico** al Google Drive y tambien al disco local de la virtual machine que esta corriendo Google Colab



In [ ]:
%%shell

mkdir -p "/content/.drive/My Drive/dm"
mkdir -p "/content/buckets"
ln -sfn "/content/.drive/My Drive/dm"   /content/buckets/b1

mkdir -p ~/.kaggle
cp /content/buckets/b1/kaggle/kaggle.json  ~/.kaggle
chmod 600 ~/.kaggle/kaggle.json


mkdir -p /content/buckets/b1/exp
mkdir -p /content/buckets/b1/datasets
mkdir -p /content/datasets


# defino funcion descargar()
descargar() {
  carpeta_destino="/content/buckets/b1/datasets/"
  url_origen="https://storage.googleapis.com/open-courses/itba2026-7c9a/dm/"
  archivo="$1"

  if ! test -f "$carpeta_destino""$archivo"; then
    wget  "$url_origen""$archivo"  -O "$carpeta_destino""$archivo"
  fi

  if ! test -f  "/content/datasets/""$archivo"; then
    cp  "$carpeta_destino""$archivo"  "/content/datasets/""$archivo"
  fi;
}


# hago la descarga efectiva, llamando a descargar()
descargar  "dataset_pequeno.csv"
descargar  "gerencial_competencia_2026.csv.gz"

## 6.3  Workflow

## Inicializacion

Esta parte se debe correr con el runtime en lenguaje **R** Ir al menu, Runtime -> Change Runtime Type -> Runtime type -> R

limpio el ambiente de R

In [ ]:
format(Sys.time(), "%a %b %d %X %Y")

In [ ]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

In [ ]:
require("data.table")

if( !require("R.utils")) install.packages("R.utils")
require("R.utils")

#### Parametros
Si es gerente, no cambie nada
<br>Si es Analista, cambie el nombre del dataset

In [ ]:
PARAM <- list()

# --- identificacion del brazo ------------------------------------------
PARAM$brazo       <- "A0_cero"
PARAM$brazo_desc  <- "Ningun tratamiento (deja los ceros)"
PARAM$experimento <- 6300

# --- semillas -----------------------------------------------------------
# 7 semillas: las 5 principales del grupo + 2 de la lista de reserva.
# El piso combinatorio del Wilcoxon pareado a dos colas es 2/2^n :
#   n=5 -> 0.0625  (NO alcanza p<0.05)
#   n=7 -> 0.0156
PARAM$semillas <- c(100043, 200063, 300089, 500069, 700021, 181219, 410341)

# semilla fija, distinta de las anteriores, para todo lo que NO es el
# LightGBM (la imputacion). Asi la unica fuente de varianza entre las 7
# corridas de un brazo es el modelo, no los datos.
PARAM$semilla_datos <- 999983

PARAM$dataset <- "gerencial_competencia_2026.csv.gz"

PARAM$ca$variables <- c(
  "internet",
  "mrentabilidad",
  "mrentabilidad_annual",
  "mcomisiones",
  "mactivos_margen",
  "mpasivos_margen",
  "mcuentas_saldo",
  "ctarjeta_visa_transacciones",
  "mtarjeta_visa_consumo",
  "mtarjeta_master_consumo",
  "ccallcenter_transacciones",
  "chomebanking_transacciones"
)

# subconjuntos por tipo, para redondear donde corresponde
PARAM$ca$binarias <- c("internet")
PARAM$ca$conteos  <- c("ctarjeta_visa_transacciones",
                       "ccallcenter_transacciones",
                       "chomebanking_transacciones")

#### Carpeta del Experimento

In [ ]:
# carpeta de trabajo

setwd("/content/buckets/b1/exp")
experimento_folder <- paste0("WF", PARAM$experimento)
dir.create(experimento_folder, showWarnings=FALSE)
setwd( paste0("/content/buckets/b1/exp/", experimento_folder ))

### 6.3.1   Preprocesamiento del dataset

#### 6.3.1.1  DT incorporar dataset

In [ ]:
# lectura del dataset
dataset <- fread(paste0("/content/datasets/", PARAM$dataset))

#### 6.3.1.2  CA  Catastrophe Analysis
Se intentan reparar las variables que para un mes están con todos los valores en cero.

El método que se utiliza es **Machine Learning** se asigna NA also valores, si ha leido bien, es la "anti imputación de valores faltantes"
<br> Usted podrá aplicar aquí otros métodos

In [ ]:
# =======================================================================
#  BRAZO A0 - NINGUN TRATAMIENTO
#  senala: NO    reconstruye: NO
# =======================================================================
# Las 12 variables de 202006 quedan tal como vienen del origen: en CERO.
# El cero es un valor real, plausible y falso: el modelo no tiene forma
# de distinguirlo de un cero legitimo. Esta celda NO modifica el dataset.

cat("Brazo A0 - sin tratamiento.\n")
cat("Variables afectadas que quedan en cero:", length(PARAM$ca$variables), "\n")

# verificacion: confirmo que efectivamente estan en cero en 202006
dataset[foto_mes == 202006,
        lapply(.SD, function(x) sum(x != 0, na.rm = TRUE)),
        .SDcols = PARAM$ca$variables]

#### 6.3.1.3  DR  Data Drifting
Se intenta corregir el data drifting, quizas ajustando por IPC ...
<br>Esta parte podrá ser abordada por todos los Analistas y también la Gerenciapero se decide pedagogicamente no incluirla en esta primer version para reducir la carga cognitiva

In [ ]:
# sin codigo en esta primera version del workflow

#### 6.3.1.3  FE_intra_manual Feature Engineering intra-mes

Agrego campos nuevos dentro del mismo mes, SIN considerar la historia.

In [ ]:
# esta funcion atributos presentes existe debido a que las modalidades poseen datasets con distinta cantidad de campos
atributos_presentes <- function( patributos )
{
  atributos <- unique( patributos )
  comun <- intersect( atributos, colnames(dataset) )

  return(  length( atributos ) == length( comun ) )
}

# el mes 1,2, ..12
if( atributos_presentes( c("foto_mes") ))
  dataset[, kmes := foto_mes %% 100]

# variable extraida de una tesis de maestria de Irlanda
if( atributos_presentes( c("mpayroll", "cliente_edad") ))
  dataset[, mpayroll_sobre_edad := mpayroll / cliente_edad]


In [ ]:
# visualizo las columas del dataset a esta etapa
colnames(dataset)

#### 6.3.1.4  FE_rf Feature Engineering de nuevas variables a partir de hojas de Random Forest

Esto se mostrará unicamente a la *modalidad Analista Sr*

In [ ]:
# No se implementa Feature Engineering a partir de Random Forest

#### 6.3.1.5  FEhist Feature Engineering historico

El Feature Engineering Histórico es la etapa que más aporta a la ganancia final, ya que enriquece cada registro del dataset con su historia.

Para cada campo del dataset original (*)
se crean lo siguientes campos de a partir de la historia
* lag1  lags de orden 1
* delta1  =  valor actual - lag1
* lag2  lags de orden 2
* delta2  = valor actual - lag2


(*) Excepto para los campos  <numero_de_cliente,  foto_mes,  clase_ternaria>

In [ ]:
# Feature Engineering Historico

# todo es lagueable, menos la primary key y la clase
cols_lagueables <- copy( setdiff(
    colnames(dataset),
    c("numero_de_cliente", "foto_mes", "clase_ternaria")
) )

# https://rdrr.io/cran/data.table/man/shift.html

# lags de orden 1
dataset[,
    paste0(cols_lagueables, "_lag1") := shift(.SD, 1, NA, "lag"),
    by = numero_de_cliente,
    .SDcols = cols_lagueables
]

# lags de orden 2
dataset[,
    paste0(cols_lagueables, "_lag2") := shift(.SD, 2, NA, "lag"),
    by = numero_de_cliente,
    .SDcols = cols_lagueables
]

# agrego los delta lags
for (vcol in cols_lagueables)
{
    dataset[, paste0(vcol, "_delta1") := get(vcol) - get(paste0(vcol, "_lag1"))]
    dataset[, paste0(vcol, "_delta2") := get(vcol) - get(paste0(vcol, "_lag2"))]
}


Verificacion de los campos recien creados

In [ ]:
ncol(dataset)
colnames(dataset)

#### 6.3.1.6  FEhist Reduccion dimensionalidad con canaritos

Esta etapa solo se mostrará a la *modalidad Anlista Sr* por algun canal secreto de forma de no confundir a los *Analista Jr*  nni distraer con detalles operativos a la estratégica *Modalidad Gerencial*

In [ ]:
# No se implementa la reduccion de la dimensionalidad con canaritos

### 6.3.2 Modelado

#### 6.3.2.1 Training Strategy

Esta etapa de Workflow de  Training Strategy esta pensada para la *Modalidad Gerencial* que posee el dataset de [202005, 202109]
<br> Si usted es un Analista, posee el periodo de [201901, 202109] y deberá experimentar en que meses le conviene experimentar

<br> A la *Modalidad Gerencial* no se le complicada la vida con el undersampling de los continua, por eso PARAM$trainingstrategy$training_pct <- 1.0
<br> Sin embargo, si usted es  *Analista SR* posee un dataset 50 veces ( filas x columnas) más grande que la *Modalidad Gerencial*  y por un tema de velocidad y experimentación más rápida puede llegar a necesitar activar el undersampling de la clase mayoritaria, a pesar de estar corriendo en Google Cloud.

Se hace una estrategia de entrenamiento muy sencilla, tomando todos los meses posibles, SIN eliminar nada x pandemia ni por ningun otro motivo

* future = 202107  obviamente completo

* final_train =  [ 202005, 202105 ]  SIN undersampling

* training
   * testing = NO HAY
   * validation =  202105   completo, sin undersampling
   * training = [ 202005, 202104 ]  donde se consideran el 100% de los CONTINUA

In [ ]:
PARAM$trainingstrategy$validate <- c(202105)

PARAM$trainingstrategy$training <- c(
  202104, 202103, 202102, 202101,
  202012, 202011, 202010, 202009, 202008, 202007,
  202006, 202005
)

PARAM$trainingstrategy$training_pct <- 1.0


PARAM$trainingstrategy$positivos <- c( "BAJA+1", "BAJA+2")

In [ ]:
# seteo la clase01   1={BAJA+1, BAJA+2}   0={CONTINUA}
dataset[, clase01 := ifelse( clase_ternaria %in% PARAM$trainingstrategy$positivos, 1, 0 )]

In [ ]:
# los campos en los que se entrena
campos_buenos <- copy( setdiff(
    colnames(dataset), c("clase_ternaria","clase01","azar"))
)

#### 6.3.2.2  Hyperparameter Tuning -- CONGELADO

En este experimento **no se corre el Grid Search**. Los hiperparametros salen de
una unica corrida previa (notebook `T4_gridsearch`) sobre el brazo A1, y se usan
identicos en los 5 brazos.

Dos razones:

1. **Costo.** El Grid Search son 60 minutos. Correrlo por brazo y por semilla
   serian 35 corridas de una hora.
2. **Ceteris paribus.** Si cada brazo optimizara sus propios hiperparametros,
   la diferencia de ganancia mezclaria el efecto del tratamiento con el efecto
   de haber caido en un punto distinto de la grilla.

Se declara en Limitaciones.

Ademas se saltean las celdas del `dtrain` / `dvalidate` (los "interminables 8
minutos"): con `training_pct = 1.0` el undersampling es un no-op, `azar` esta
excluido de `campos_buenos`, y `dtrain` solo servia para alimentar el Grid
Search. El modelo final no los usa.

In [ ]:
if (!require("lightgbm")) install.packages("lightgbm")
require("lightgbm")

# parametros fijos del LightGBM (identicos al workflow original)
PARAM$lgbm$param_fijos <- list(
  objective         = "binary",
  metric            = "auc",
  first_metric_only = TRUE,
  boost_from_average = TRUE,
  feature_pre_filter = FALSE,
  verbosity         = -100,
  force_row_wise    = TRUE,
  seed              = NA_integer_,   # lo pisa el loop de semillas
  max_bin           = 31,
  learning_rate     = 0.03,
  feature_fraction  = 0.5
)

# =======================================================================
#  HIPERPARAMETROS CONGELADOS  -- salida del notebook T4_gridsearch
# =======================================================================
#  OJO: son TRES, no dos. La celda 68 del workflow original saca
#  num_iterations de aca, no de param_fijos.
#  Corrida del 2026-09-07, WF6390, semilla 100043.
#  AUC en validation = 0.9540665
PARAM$out$lgbm$mejores_hiperparametros <- list(
  num_leaves       = 256L,
  min_data_in_leaf = 64L,
  num_iterations   = 428L
)

if (anyNA(unlist(PARAM$out$lgbm$mejores_hiperparametros))) {
  stop("FALTAN los hiperparametros congelados. Corre primero el notebook ",
       "T4_gridsearch y pega aca los tres valores que imprime.")
}

PARAM$out$lgbm$mejores_hiperparametros

### 6.3.3 Produccion

#### Final Training
Construyo el modelo final, que es uno solo, no hace ningun tipo de particion < training, validation, testing>]

##### Final Training Dataset

Aqui esta la gran decision de en qué meses hago el Final Training
<br> debo utilizar los mejores hiperparámetros que encontré en la optimización de hiperparámetros

In [ ]:
PARAM$trainingstrategy$final_train <- c(
  202105, 202104, 202103, 202102, 202101,
  202012, 202011, 202010, 202009, 202008, 202007,
  202006, 202005
)

dataset[, fold_final_train := foto_mes %in% PARAM$trainingstrategy$final_train ]

# creo el dfinal_train en formato  LightGBM
dfinal_train <- lgb.Dataset(
  data= data.matrix(dataset[fold_final_train == TRUE, campos_buenos, with= FALSE]),
  label= dataset[fold_final_train == TRUE, clase01],
  free_raw_data= TRUE
)

nrow( dfinal_train) # verifico el tamaño

#### Scoring

Aplico el modelo final a los datos del futuro

In [ ]:
PARAM$trainingstrategy$future <- c(202107)

dfuture <- dataset[ foto_mes %in% PARAM$trainingstrategy$future ]

#### Loop de semillas

`dataset`, `dfinal_train` y `dfuture` ya estan construidos y **no se vuelven a
tocar**. La unica cosa que cambia entre las 7 corridas es `param_final$seed`.

Cada iteracion deja en la carpeta del experimento:

| archivo | que es |
|---|---|
| `modelo_<semilla>.txt` | el modelo LightGBM |
| `impo_<semilla>.txt` | importancia de variables |
| `prediccion_<semilla>.txt` | probabilidades sobre 202107 |
| `ganancias_<semilla>.txt` | la curva de ganancia completa |
| `curva_de_ganancia_<semilla>.pdf` | el grafico |
| `PARAM_<semilla>.yml` | **la ganancia a reportar** |
| `resultados_<brazo>.txt` | acumulador, una fila por semilla |

El acumulador se reescribe **despues de cada semilla**, no al final: si Colab se
desconecta en la semilla 5, las 4 anteriores quedan guardadas.

In [ ]:
if (!require("yaml")) install.packages("yaml")
require("yaml")

fijos <- copy(PARAM$lgbm$param_fijos)
fijos$num_iterations        <- NULL
fijos$early_stopping_rounds <- NULL

tb_resultados <- data.table(
  brazo                  = character(),
  semilla                = integer(),
  ganancia_suavizada_max = numeric(),
  envios                 = integer()
)

archivo_resultados <- paste0("resultados_", PARAM$brazo, ".txt")

for (semilla in PARAM$semillas) {

  cat("\n=== ", format(Sys.time(), "%X"), " brazo ", PARAM$brazo,
      "  semilla ", semilla, " ===\n", sep = "")

  param_final <- c(fijos, PARAM$out$lgbm$mejores_hiperparametros)
  param_final$seed <- semilla

  # -- final training ---------------------------------------------------
  # dfinal_train se construye UNA sola vez y se reusa. Si alguna version
  # de lightgbm se queja por el free_raw_data, lo reconstruyo y sigo.
  final_model <- tryCatch(
    lgb.train(data = dfinal_train, param = param_final, verbose = -100),
    error = function(e) {
      message("  reconstruyo dfinal_train: ", conditionMessage(e))
      dfinal_train <<- lgb.Dataset(
        data  = data.matrix(dataset[fold_final_train == TRUE, campos_buenos, with = FALSE]),
        label = dataset[fold_final_train == TRUE, clase01],
        free_raw_data = TRUE)
      lgb.train(data = dfinal_train, param = param_final, verbose = -100)
    })

  lgb.save(final_model, filename = paste0("modelo_", semilla, ".txt"))

  fwrite(as.data.table(lgb.importance(final_model)),
         file = paste0("impo_", semilla, ".txt"), sep = "\t")

  # -- scoring ----------------------------------------------------------
  prediccion <- predict(final_model,
                        data.matrix(dfuture[, campos_buenos, with = FALSE]))

  tb_prediccion <- dfuture[, list(numero_de_cliente)]
  tb_prediccion[, prob := prediccion]
  fwrite(tb_prediccion, file = paste0("prediccion_", semilla, ".txt"), sep = "\t")

  # -- curva de ganancia ------------------------------------------------
  tb_prediccion[, clase_ternaria := dfuture$clase_ternaria]
  tb_prediccion[, ganancia := -0.025]
  tb_prediccion[clase_ternaria == "BAJA+2", ganancia := 0.975]

  setorder(tb_prediccion, -prob)
  tb_prediccion[, gan_acum := cumsum(ganancia)]
  tb_prediccion[, gan_suavizada := frollmean(x = gan_acum, n = 400,
                    align = "center", na.rm = TRUE, hasNA = TRUE)]

  resultado <- list()
  resultado$ganancia_suavizada_max <- max(tb_prediccion$gan_suavizada, na.rm = TRUE)
  resultado$envios <- which.max(tb_prediccion$gan_suavizada)

  fwrite(tb_prediccion, file = paste0("ganancias_", semilla, ".txt"), sep = "\t")

  # -- grafico ----------------------------------------------------------
  tb_prediccion[, envios := .I]
  pdf(paste0("curva_de_ganancia_", semilla, ".pdf"))
  plot(x = tb_prediccion$envios, y = tb_prediccion$gan_acum,
       type = "l", col = "gray", xlim = c(0, 6000), ylim = c(0, 50),
       main = paste0(PARAM$brazo, "  sem=", semilla,
                     "  gan=", as.integer(resultado$ganancia_suavizada_max),
                     "  envios=", resultado$envios),
       xlab = "Envios", ylab = "Ganancia", panel.first = grid())
  dev.off()

  # -- PARAM.yml de esta corrida ----------------------------------------
  PARAM_corrida <- copy(PARAM)
  PARAM_corrida$semillas <- NULL
  PARAM_corrida$semilla_de_esta_corrida <- semilla
  PARAM_corrida$lgbm$param_fijos$seed   <- semilla   # que quede en el registro
  PARAM_corrida$resultado <- resultado
  write_yaml(PARAM_corrida, file = paste0("PARAM_", semilla, ".yml"))

  # -- acumulador, se reescribe en cada vuelta ---------------------------
  tb_resultados <- rbind(tb_resultados, data.table(
    brazo                  = PARAM$brazo,
    semilla                = as.integer(semilla),
    ganancia_suavizada_max = resultado$ganancia_suavizada_max,
    envios                 = as.integer(resultado$envios)))

  fwrite(tb_resultados, file = archivo_resultados, sep = "\t")

  cat("   ganancia_suavizada_max = ", resultado$ganancia_suavizada_max,
      "   envios = ", resultado$envios, "\n", sep = "")

  rm(final_model, tb_prediccion, prediccion)
  gc(full = TRUE, verbose = FALSE)
}

tb_resultados

In [ ]:
format(Sys.time(), "%a %b %d %X %Y")